In [1]:
import os
from pathlib import Path
os.chdir(Path().resolve().parent)

import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from data_loader.data_loaders import label_encoding, get_dataloader, get_within_decade_split, get_transfer_split, transform
from models.cnn import PosterCNN

In [3]:
import yaml
with open("configs/default.yaml") as f:
    config = yaml.safe_load(f)

In [5]:
df = pd.read_csv(config["data"]["metadata_path"])

label_to_idx, idx_to_label = label_encoding(df)

num_classes = len(label_to_idx)

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
print(f'Classes: {list(label_to_idx.keys())}')
print(f'Total samples: {len(df)}')

Classes: ['Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'History', 'Horror', 'Music', 'Mystery', 'Romance', 'Science Fiction', 'TV Movie', 'Thriller', 'War', 'Western']
Total samples: 85555


In [8]:
def load_and_evaluate(exp_name, test_df):
    model = PosterCNN(num_classes=num_classes)
    model.load_state_dict(torch.load(f'checkpoints/{exp_name}_best.pt', weights_only=True, map_location=device))

    model = model.to(device)
    model.eval()

    loader = get_dataloader(test_df, label_to_idx, transform, batch_size=64, shuffle=False, num_workers=0)

    all_predictions = []
    
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            predictions = model(images).argmax(dim=1).cpu()
            all_predictions.append(predictions)
            all_labels.append(labels)

    return torch.cat(all_predictions).numpy(), torch.cat(all_labels).numpy()

In [9]:
experiments = {
    "within_1980s": get_within_decade_split(df, 1980),
    "within_2010s": get_within_decade_split(df, 2010),
    "forward_1980s_1990s": get_transfer_split(df, 1980, 1990),
    "large_gap_1980s_2020s": get_transfer_split(df, 1980, 2020),
    "backward_2010s_1980s": get_transfer_split(df, 2010, 1980),
}

Within 1980s | Train: 15376 | Val: 1922 | Test: 1922
Within 2010s | Train: 14216 | Val: 1778 | Test: 1778
Transfer 1980s -> 1990s | Train: 15376 | Val: 3844 | Test: 22489
Transfer 1980s -> 2020s | Train: 15376 | Val: 3844 | Test: 4390
Transfer 2010s -> 1980s | Train: 14217 | Val: 3555 | Test: 19220


In [ ]:
results = {}